In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# 현재 작업 디렉토리 확인 및 변경
notebook_dir = r'C:\Users\tkdwl\Desktop\토스 경진대회'
os.chdir(notebook_dir)

# 데이터 로드
df_train = pd.read_parquet('./train.parquet')
df_test = pd.read_parquet('./test.parquet')

print(f"df_train shape: {df_train.shape}")
print(f"df_test shape: {df_test.shape}")

### 1. 결측값 처리 (5% 이상 컬럼 제거 + 행 제거 + impute)

In [2]:
target_col = 'clicked'
id_col = 'ID'
col_drop_threshold = 0.05

# 결측 비율 5% 이상 컬럼 제거
missing_train_ratio = df_train.isnull().mean()
missing_test_ratio = df_test.isnull().mean()
high_missing_cols = sorted(
    set(missing_train_ratio[missing_train_ratio >= col_drop_threshold].index) |
    set(missing_test_ratio[missing_test_ratio >= col_drop_threshold].index)
)
high_missing_cols = [col for col in high_missing_cols if col not in [target_col, id_col]]

if high_missing_cols:
    df_train = df_train.drop(columns=high_missing_cols)
    df_test = df_test.drop(columns=high_missing_cols)

# 수치형/범주형 분리
num_types = ['float32', 'int32', 'float64', 'int64']
train_num_cols = [col for col in df_train.select_dtypes(include=num_types).columns if col != target_col]
test_num_cols = [col for col in df_test.select_dtypes(include=num_types).columns]

train_cat_cols = df_train.select_dtypes(include=['object']).columns.tolist()
test_cat_cols = df_test.select_dtypes(include=['object']).columns.tolist()
if id_col in train_cat_cols: train_cat_cols.remove(id_col)
if id_col in test_cat_cols: test_cat_cols.remove(id_col)
if target_col in train_cat_cols: train_cat_cols.remove(target_col)

# 수치형 결측 행 제거
train_missing_mask = df_train[train_num_cols].isnull().any(axis=1)
test_missing_mask = df_test[test_num_cols].isnull().any(axis=1)
df_train = df_train.loc[~train_missing_mask].reset_index(drop=True)
df_test = df_test.loc[~test_missing_mask].reset_index(drop=True)

# 범주형 impute (최빈값)
from sklearn.impute import SimpleImputer
cat_imputer = SimpleImputer(strategy='most_frequent')
common_cats = sorted(set(train_cat_cols) & set(test_cat_cols))
if common_cats:
    df_train[common_cats] = cat_imputer.fit_transform(df_train[common_cats])
    df_test[common_cats] = cat_imputer.transform(df_test[common_cats])

print(f"최종 결측값: train={df_train.isnull().sum().sum()}, test={df_test.isnull().sum().sum()}")

### 2. 피처 엔지니어링

In [3]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans

# hour 문자열 → 정수
df_train['hour'] = pd.to_numeric(df_train['hour'], errors='coerce')
df_test['hour'] = pd.to_numeric(df_test['hour'], errors='coerce')
median_hour = df_train['hour'].median()
df_train['hour'] = df_train['hour'].fillna(median_hour)
df_test['hour'] = df_test['hour'].fillna(median_hour)

# 시간 피처
for df in [df_train, df_test]:
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

# seq 피처
def process_seq(df, max_items=50, top_n=20):
    df['seq_length'] = df['seq'].str.split(',').str.len()
    df['seq_unique'] = df['seq'].str.split(',').apply(lambda x: len(set(x)) if isinstance(x, list) else 0)
    df['seq_diversity'] = df['seq_unique'] / (df['seq_length'] + 1e-8)
    
    all_items = [item.strip() for seq in df['seq'].str.split(',') for item in seq[:max_items]]
    top_items = pd.Series(all_items).value_counts().head(100).index
    for item in top_items[:top_n]:
        df[f'seq_has_{item}'] = df['seq'].str.contains(item, regex=False).astype(int)
    return df

df_train = process_seq(df_train)
df_test = process_seq(df_test)

# 클러스터링
cluster_cols = [col for col in train_num_cols if col not in ['clicked']]
scaler = StandardScaler()
scaled_train = scaler.fit_transform(df_train[cluster_cols])
scaled_test = scaler.transform(df_test[cluster_cols])

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df_train['cluster'] = kmeans.fit_predict(scaled_train)
df_test['cluster'] = kmeans.predict(scaled_test)

# Label Encoding (train만 fit)
cat_cols = ['gender', 'age_group', 'inventory_id', 'day_of_week', 'cluster']
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_train[col] = df_train[col].astype(str)
    df_test[col] = df_test[col].astype(str)
    le.fit(df_train[col])
    df_train[col] = le.transform(df_train[col])
    df_test[col] = df_test[col].apply(lambda x: le.transform([x])[0] if x in le.classes_ else -1)
    label_encoders[col] = le

# 스케일링 (test도 포함)
num_cols = [col for col in train_num_cols if col != 'clicked']
df_train[num_cols] = scaler.fit_transform(df_train[num_cols])
df_test[num_cols] = scaler.transform(df_test[num_cols])

### 3. 데이터 분리 + SMOTE

In [4]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

feature_cols = [col for col in df_train.columns if col not in [target_col, id_col]]
X = df_train[feature_cols]
y = df_train[target_col]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# age_click_mean (train 기준)
age_mean = X_train.groupby('age_group').size()
age_click = X_train.groupby('age_group')[y_train.name].mean()
X_train['age_click_mean'] = X_train['age_group'].map(age_click)
X_valid['age_click_mean'] = X_valid['age_group'].map(age_click).fillna(y_train.mean())
df_test['age_click_mean'] = df_test['age_group'].map(age_click).fillna(y_train.mean())

# SMOTE
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"SMOTE 후: {X_train_res.shape}, click rate: {y_train_res.mean():.3f}")

### 4. 전처리 결과 저장 및 불러오기

In [ ]:
import joblib

preprocessed_dir = os.path.join(notebook_dir, 'preprocessed_data')
os.makedirs(preprocessed_dir, exist_ok=True)

preprocessed_path = os.path.join(preprocessed_dir, 'dataset.pkl')

joblib.dump(
    {
        'X_train_res': X_train_res,
        'y_train_res': y_train_res,
        'X_valid': X_valid,
        'y_valid': y_valid,
        'df_test': df_test,
        'feature_cols': feature_cols
    },
    preprocessed_path
)

print(f"전처리 데이터 저장 완료: {preprocessed_path}")


In [ ]:
preprocessed_dir = os.path.join(notebook_dir, 'preprocessed_data')
preprocessed_path = os.path.join(preprocessed_dir, 'dataset.pkl')

loaded_data = joblib.load(preprocessed_path)

X_train_res = loaded_data['X_train_res']
y_train_res = loaded_data['y_train_res']
X_valid = loaded_data['X_valid']
y_valid = loaded_data['y_valid']
df_test = loaded_data['df_test']
feature_cols = loaded_data['feature_cols']

print("전처리 데이터 로드 완료!")


### 5. LightGBM 모델 학습 + 예측

In [5]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

lgb_train = lgb.Dataset(X_train_res, label=y_train_res)
lgb_valid = lgb.Dataset(X_valid, label=y_valid, reference=lgb_train)

params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 64,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1
}

model = lgb.train(
    params,
    lgb_train,
    num_boost_round=2000,
    valid_sets=[lgb_train, lgb_valid],
    early_stopping_rounds=100,
    verbose_eval=100
)

# Valid AUC
pred_valid = model.predict(X_valid)
auc = roc_auc_score(y_valid, pred_valid)
print(f"\nValidation AUC: {auc:.5f}")

### 6. 제출 파일 생성

In [6]:
X_test = df_test[feature_cols]
pred_test = model.predict(X_test)

submission = pd.DataFrame({
    'ID': df_test[id_col],
    'clicked': pred_test
})
submission.to_csv('submission.csv', index=False)
print("submission.csv 저장 완료!")